In [ ]:
import backtrader as bt
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

class MultiAssetTripleStrategy(bt.Strategy):
    params = (
        ('first_long_buy', 13),
        ('first_long_sell', 16),
        ('short_entry', 18),
        ('short_exit', 20),
        ('second_long_buy', 27),
        ('second_long_sell', 2),
        ('printlog', False),
    )
    
    def __init__(self):
        self.orders = {}
        self.position_types = {}
        self.entry_months = {}
        self.entry_prices = {}
        self.first_long_months = {}
        self.short_months = {}
        self.second_long_months = {}
        
        # Inizializza per ogni data feed
        for data in self.datas:
            name = data._name
            self.orders[name] = None
            self.position_types[name] = None
            self.entry_months[name] = None
            self.entry_prices[name] = 0
            self.first_long_months[name] = None
            self.short_months[name] = None
            self.second_long_months[name] = None
        
        # Per tracciare l'equity
        self.equity_curve = []
        self.dates = []
        
        # Traccia trade per asset
        self.trades = {}
        for data in self.datas:
            self.trades[data._name] = {
                'first_long': [],
                'short': [],
                'second_long': []
            }

    def notify_order(self, order):
        """Gestisce lo stato degli ordini"""
        if order.status in [order.Completed]:
            data_name = order.data._name
            current_date = order.data.datetime.date(0)
            current_day = current_date.day
            
            if order.isbuy():
                self.entry_prices[data_name] = order.executed.price
                self.entry_months[data_name] = (current_date.year, current_date.month)
                
                if self.position_types[data_name] == 'short':
                    # Chiusura short
                    pnl = (self.entry_prices[data_name] - order.executed.price) * order.executed.size
                    self.log(f'{data_name} 🟦 COVER SHORT @ {order.executed.price:.2f}, P&L: ${pnl:.2f}')
                    self.trades[data_name]['short'].append({
                        'pnl': pnl,
                        'exit_date': current_date
                    })
                    self.position_types[data_name] = None
                    self.entry_months[data_name] = None
                else:
                    # Apertura long
                    if self.params.first_long_buy <= current_day < self.params.short_entry:
                        self.position_types[data_name] = 'first_long'
                        self.log(f'{data_name} 🟢 LONG1 @ {order.executed.price:.2f}')
                    else:
                        self.position_types[data_name] = 'second_long'
                        self.log(f'{data_name} 🟢 LONG2 @ {order.executed.price:.2f}')
                    
            elif order.issell():
                exit_date = current_date
                
                if self.position_types[data_name] == 'first_long':
                    pnl = (order.executed.price - self.entry_prices[data_name]) * order.executed.size
                    self.log(f'{data_name} 🔵 SELL LONG1, P&L: ${pnl:.2f}')
                    self.trades[data_name]['first_long'].append({
                        'pnl': pnl,
                        'exit_date': exit_date
                    })
                    self.position_types[data_name] = None
                    self.entry_months[data_name] = None
                    
                elif self.position_types[data_name] == 'second_long':
                    pnl = (order.executed.price - self.entry_prices[data_name]) * order.executed.size
                    self.log(f'{data_name} 🔵 SELL LONG2, P&L: ${pnl:.2f}')
                    self.trades[data_name]['second_long'].append({
                        'pnl': pnl,
                        'exit_date': exit_date
                    })
                    self.position_types[data_name] = None
                    self.entry_months[data_name] = None
                    
                else:
                    # Apertura short
                    self.entry_prices[data_name] = order.executed.price
                    self.entry_months[data_name] = (current_date.year, current_date.month)
                    self.position_types[data_name] = 'short'
                    self.log(f'{data_name} 🔴 OPEN SHORT @ {order.executed.price:.2f}')
            
            self.orders[data_name] = None
            
        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            data_name = order.data._name
            self.orders[data_name] = None

    def log(self, txt, dt=None):
        """Logging function"""
        if self.params.printlog:
            if dt is None:
                dt = self.datas[0].datetime.date(0)
            print(f'{dt.isoformat()} {txt}')

    def next(self):
        # Traccia equity
        self.dates.append(self.datas[0].datetime.date(0))
        self.equity_curve.append(self.broker.getvalue())
        
        # Processa ogni asset
        for data in self.datas:
            data_name = data._name
            
            if self.orders[data_name]:
                continue
            
            current_date = data.datetime.date(0)
            current_day = current_date.day
            current_month = (current_date.year, current_date.month)
            
            # Reset tracker all'inizio del mese
            if current_day < self.params.first_long_buy:
                self.first_long_months[data_name] = None
                self.short_months[data_name] = None
                self.second_long_months[data_name] = None
            
            # ======= CHIUSURA POSIZIONI =======
            pos = self.getposition(data)
            
            if self.position_types[data_name] == 'first_long':
                if current_day >= self.params.first_long_sell and current_month == self.entry_months[data_name]:
                    self.orders[data_name] = self.close(data=data)
                    
            elif self.position_types[data_name] == 'short':
                if current_day >= self.params.short_exit and current_month == self.entry_months[data_name]:
                    self.orders[data_name] = self.close(data=data)
                    
            elif self.position_types[data_name] == 'second_long':
                if current_month != self.entry_months[data_name] and current_day >= self.params.second_long_sell:
                    self.orders[data_name] = self.close(data=data)
            
            # ======= APERTURA POSIZIONI =======
            elif not pos:
                cash = self.broker.getcash()
                price = data.close[0]
                # Divide il cash equamente tra tutti gli asset
                size = int((cash * 0.99 / len(self.datas)) / price)
                
                if size > 0:
                    # PRIMO LONG: 13-15
                    if (self.params.first_long_buy <= current_day < self.params.first_long_sell and 
                        self.first_long_months[data_name] != current_month):
                        self.orders[data_name] = self.buy(data=data, size=size)
                        self.first_long_months[data_name] = current_month
                    
                    # SHORT: 18-19
                    elif (self.params.short_entry <= current_day < self.params.short_exit and 
                          self.short_months[data_name] != current_month):
                        self.orders[data_name] = self.sell(data=data, size=size)
                        self.short_months[data_name] = current_month
                    
                    # SECONDO LONG: 27+
                    elif (current_day >= self.params.second_long_buy and 
                          self.second_long_months[data_name] != current_month):
                        self.orders[data_name] = self.buy(data=data, size=size)
                        self.second_long_months[data_name] = current_month


def plot_comprehensive_analysis(dates, equity, metrics, trades, filename='multi_asset_analysis.jpg'):
    """Crea analisi completa multi-asset"""
    
    fig = plt.figure(figsize=(22, 16))
    gs = fig.add_gridspec(4, 5, height_ratios=[2.5, 1, 1.5, 1.5], hspace=0.4, wspace=0.4)
    
    ax_equity = fig.add_subplot(gs[0, :])
    ax_dd = fig.add_subplot(gs[1, :])
    ax_comparison = fig.add_subplot(gs[2, :])
    
    # Asset individual bars (9 assets in bottom row)
    asset_axes = []
    for i in range(9):
        if i < 5:
            ax = fig.add_subplot(gs[3, i])
        else:
            # Seconda riga se necessario
            ax = fig.add_subplot(gs[2, i-5]) if i >= 5 else None
    
    # ===== EQUITY CURVE =====
    ax_equity.plot(dates, equity, linewidth=3, color='#1f77b4', label='Portfolio Equity', zorder=2)
    ax_equity.fill_between(dates, equity, alpha=0.3, color='#1f77b4', zorder=1)
    
    ax_equity.set_title('Multi-Asset Portfolio (SPY + QQQ + Mag7) - Triple Strategy\n(Long 13→16 | Short 18→20 | Long 27→2)', 
                        fontsize=18, fontweight='bold', pad=20)
    ax_equity.set_ylabel('Portfolio Value ($)', fontsize=13, fontweight='bold')
    ax_equity.grid(True, alpha=0.3, linestyle='--')
    ax_equity.legend(loc='upper left', fontsize=12)
    ax_equity.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    # ===== DRAWDOWN =====
    equity_series = pd.Series(equity, index=dates)
    rolling_max = equity_series.expanding().max()
    drawdown = ((equity_series - rolling_max) / rolling_max) * 100
    
    ax_dd.fill_between(dates, drawdown, 0, color='red', alpha=0.3)
    ax_dd.plot(dates, drawdown, color='darkred', linewidth=2, label='Drawdown')
    ax_dd.set_ylabel('Drawdown (%)', fontsize=12, fontweight='bold')
    ax_dd.grid(True, alpha=0.3, linestyle='--')
    ax_dd.legend(loc='lower right', fontsize=10)
    
    # ===== ASSET COMPARISON =====
    assets = list(trades.keys())
    asset_pnls = []
    asset_returns = []
    
    for asset in assets:
        total_pnl = (sum(t['pnl'] for t in trades[asset]['first_long']) +
                     sum(t['pnl'] for t in trades[asset]['short']) +
                     sum(t['pnl'] for t in trades[asset]['second_long']))
        asset_pnls.append(total_pnl)
        
        # Calcola return % assumendo allocazione equa
        initial_per_asset = metrics['initial_capital'] / len(assets)
        ret_pct = (total_pnl / initial_per_asset) * 100
        asset_returns.append(ret_pct)
    
    # Bar chart comparison
    colors = ['green' if x > 0 else 'red' for x in asset_pnls]
    bars = ax_comparison.bar(assets, asset_pnls, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Aggiungi etichette sui bar
    for i, (bar, pnl, ret) in enumerate(zip(bars, asset_pnls, asset_returns)):
        height = bar.get_height()
        ax_comparison.text(bar.get_x() + bar.get_width()/2., height,
                          f'${pnl:,.0f}\n({ret:+.1f}%)',
                          ha='center', va='bottom' if height > 0 else 'top',
                          fontsize=9, fontweight='bold')
    
    ax_comparison.axhline(y=0, color='black', linestyle='-', linewidth=1.5)
    ax_comparison.set_title('Asset-by-Asset P&L Comparison', fontsize=14, fontweight='bold', pad=15)
    ax_comparison.set_ylabel('Total P&L ($)', fontsize=12, fontweight='bold')
    ax_comparison.set_xlabel('Asset', fontsize=12, fontweight='bold')
    ax_comparison.grid(True, alpha=0.3, axis='y')
    ax_comparison.tick_params(axis='x', labelsize=11, rotation=0)
    
    # ===== INDIVIDUAL ASSET BREAKDOWN =====
    asset_axes = [fig.add_subplot(gs[3, i]) for i in range(5)]
    asset_axes += [fig.add_subplot(gs[2, i]) for i in range(4)]  # Seconda riga
    
    for i, (asset, ax) in enumerate(zip(assets, asset_axes)):
        long1_pnl = sum(t['pnl'] for t in trades[asset]['first_long'])
        short_pnl = sum(t['pnl'] for t in trades[asset]['short'])
        long2_pnl = sum(t['pnl'] for t in trades[asset]['second_long'])
        
        categories = ['L1\n13-16', 'S\n18-20', 'L2\n27-2']
        values = [long1_pnl, short_pnl, long2_pnl]
        colors_breakdown = ['green' if v > 0 else 'red' for v in values]
        
        ax.bar(categories, values, color=colors_breakdown, alpha=0.7, edgecolor='black')
        ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
        ax.set_title(f'{asset}', fontsize=10, fontweight='bold')
        ax.set_ylabel('P&L ($)', fontsize=8)
        ax.tick_params(labelsize=8)
        ax.grid(True, alpha=0.3, axis='y')
        
        # Aggiungi totale
        total = sum(values)
        ax.text(0.5, 0.95, f'Tot: ${total:,.0f}', 
               transform=ax.transAxes, ha='center', va='top',
               fontsize=9, fontweight='bold',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Format X-axis
    for ax in [ax_equity, ax_dd]:
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        ax.xaxis.set_major_locator(mdates.YearLocator(2))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # ===== METRICS BOX =====
    total_pnl = sum(asset_pnls)
    best_asset = assets[asset_pnls.index(max(asset_pnls))]
    worst_asset = assets[asset_pnls.index(min(asset_pnls))]
    
    textstr = f"""
    PORTFOLIO METRICS
    {'─' * 32}
    Initial: ${metrics['initial_capital']:,.0f}
    Final: ${metrics['final_value']:,.0f}
    P&L: ${total_pnl:,.0f}
    Return: {metrics['total_return']:.2f}%
    CAGR: {metrics['cagr']:.2f}%
    
    Max DD: {metrics['max_dd']:.2f}%
    Sharpe: {metrics['sharpe']:.2f}
    Win Rate: {metrics['win_rate']:.1f}%
    
    Best: {best_asset} ${max(asset_pnls):,.0f}
    Worst: {worst_asset} ${min(asset_pnls):,.0f}
    
    Assets: {len(assets)}
    Allocation: {100/len(assets):.1f}% each
    """
    
    props = dict(boxstyle='round', facecolor='lightblue', alpha=0.95, edgecolor='black', linewidth=2)
    ax_equity.text(0.015, 0.97, textstr, transform=ax_equity.transAxes, fontsize=10,
                  verticalalignment='top', bbox=props, family='monospace')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✓ Grafico salvato: '{filename}'")
    plt.close()


def calculate_metrics(strat, initial_cash, final_value, years):
    """Calcola tutte le metriche"""
    trade_analysis = strat.analyzers.trade.get_analysis()
    drawdown_analysis = strat.analyzers.drawdown.get_analysis()
    sharpe_analysis = strat.analyzers.sharpe.get_analysis()
    
    total_trades = trade_analysis.total.closed if hasattr(trade_analysis, 'total') else 0
    won_trades = trade_analysis.won.total if hasattr(trade_analysis, 'won') else 0
    
    win_rate = (won_trades / total_trades * 100) if total_trades > 0 else 0
    total_return = ((final_value / initial_cash) - 1) * 100
    cagr = ((final_value / initial_cash) ** (1 / years) - 1) * 100
    max_dd = drawdown_analysis.max.drawdown if hasattr(drawdown_analysis, 'max') else 0
    sharpe = sharpe_analysis.get('sharperatio', 0) if sharpe_analysis else 0
    
    return {
        'initial_capital': initial_cash,
        'final_value': final_value,
        'total_return': total_return,
        'cagr': cagr,
        'max_dd': max_dd,
        'sharpe': sharpe,
        'total_trades': total_trades,
        'won_trades': won_trades,
        'win_rate': win_rate,
    }


def print_detailed_analysis(metrics, trades):
    """Stampa analisi dettagliata"""
    print("\n" + "="*80)
    print(" " * 20 + "MULTI-ASSET PORTFOLIO ANALYSIS")
    print(" " * 18 + "(SPY + QQQ + Magnificent 7)")
    print("="*80)
    
    print(f"\n{'PORTFOLIO OVERVIEW':.<50}")
    print(f"  Initial Capital: {metrics['initial_capital']:>35,.2f} $")
    print(f"  Final Value: {metrics['final_value']:>39,.2f} $")
    print(f"  Total P&L: {metrics['final_value'] - metrics['initial_capital']:>41,.2f} $")
    print(f"  Total Return: {metrics['total_return']:>38.2f} %")
    print(f"  CAGR: {metrics['cagr']:>48.2f} %")
    print(f"  Max Drawdown: {metrics['max_dd']:>38.2f} %")
    print(f"  Sharpe Ratio: {metrics['sharpe']:>40.2f}")
    print(f"  Win Rate: {metrics['win_rate']:>42.2f} %")
    
    # Analisi per asset
    print(f"\n{'='*80}")
    print(f"{'ASSET BREAKDOWN':^80}")
    print(f"{'='*80}")
    
    # Header
    print(f"\n{'Asset':<10} {'Long1 P&L':>12} {'Short P&L':>12} {'Long2 P&L':>12} {'Total P&L':>12} {'Return %':>10}")
    print("-" * 80)
    
    asset_totals = []
    for asset in sorted(trades.keys()):
        long1_pnl = sum(t['pnl'] for t in trades[asset]['first_long'])
        short_pnl = sum(t['pnl'] for t in trades[asset]['short'])
        long2_pnl = sum(t['pnl'] for t in trades[asset]['second_long'])
        total_pnl = long1_pnl + short_pnl + long2_pnl
        
        # Calcola return % per asset
        initial_per_asset = metrics['initial_capital'] / len(trades)
        ret_pct = (total_pnl / initial_per_asset) * 100
        
        asset_totals.append((asset, total_pnl, ret_pct))
        
        print(f"{asset:<10} ${long1_pnl:>11,.0f} ${short_pnl:>11,.0f} ${long2_pnl:>11,.0f} ${total_pnl:>11,.0f} {ret_pct:>9.1f}%")
    
    print("-" * 80)
    
    # Totali
    total_long1 = sum(sum(t['pnl'] for t in trades[a]['first_long']) for a in trades)
    total_short = sum(sum(t['pnl'] for t in trades[a]['short']) for a in trades)
    total_long2 = sum(sum(t['pnl'] for t in trades[a]['second_long']) for a in trades)
    total_all = total_long1 + total_short + total_long2
    
    print(f"{'TOTAL':<10} ${total_long1:>11,.0f} ${total_short:>11,.0f} ${total_long2:>11,.0f} ${total_all:>11,.0f} {metrics['total_return']:>9.1f}%")
    
    # Top/Bottom performers
    asset_totals.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n{'TOP PERFORMERS':.<50}")
    for i, (asset, pnl, ret) in enumerate(asset_totals[:3], 1):
        print(f"  {i}. {asset:<8} ${pnl:>15,.0f}  ({ret:>+6.1f}%)")
    
    print(f"\n{'BOTTOM PERFORMERS':.<50}")
    for i, (asset, pnl, ret) in enumerate(asset_totals[-3:], 1):
        print(f"  {i}. {asset:<8} ${pnl:>15,.0f}  ({ret:>+6.1f}%)")
    
    # Strategy breakdown
    print(f"\n{'='*80}")
    print(f"{'STRATEGY BREAKDOWN':^80}")
    print(f"{'='*80}")
    
    print(f"\n  Long 1 (13→16):  ${total_long1:>15,.0f}  ({total_long1/total_all*100:>5.1f}%)")
    print(f"  Short (18→20):   ${total_short:>15,.0f}  ({total_short/total_all*100:>5.1f}%)")
    print(f"  Long 2 (27→2):   ${total_long2:>15,.0f}  ({total_long2/total_all*100:>5.1f}%)")
    
    # Trade counts
    total_trades_count = sum(len(trades[a]['first_long']) + len(trades[a]['short']) + len(trades[a]['second_long']) for a in trades)
    
    print(f"\n{'TRADE STATISTICS':.<50}")
    print(f"  Total Trades: {total_trades_count:>38}")
    print(f"  Trades per Asset: {total_trades_count/len(trades):>33.1f}")
    print(f"  Assets Traded: {len(trades):>37}")
    
    print("\n" + "="*80 + "\n")


if __name__ == '__main__':
    cerebro = bt.Cerebro()
    cerebro.addstrategy(MultiAssetTripleStrategy)

    # Lista asset: SPY, QQQ + Magnificent 7
    ASSETS = ['SPY', 'QQQ', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'TSLA', 'META',]
    
    print("="*80)
    print(" " * 20 + "DOWNLOADING MULTI-ASSET DATA")
    print("="*80)
    
    dataframes = {}
    for ticker in ASSETS:
        print(f"Downloading {ticker}...", end=' ')
        try:
            df = yf.download(ticker, 
                           start='2010-01-01', 
                           end='2025-09-29', 
                           auto_adjust=False,
                           progress=False)
            
            # Pulisci colonne
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df.columns = [col.lower().replace(' ', '') for col in df.columns]
            df.dropna(inplace=True)
            
            dataframes[ticker] = df
            print(f"✓ {len(df)} rows")
        except Exception as e:
            print(f"✗ Error: {e}")
    
    # Aggiungi data feed per ogni asset
    for ticker in ASSETS:
        if ticker in dataframes:
            data = bt.feeds.PandasData(dataname=dataframes[ticker], name=ticker)
            cerebro.adddata(data)
    
    print(f"\n✓ Loaded {len(dataframes)} assets successfully")

    # Configurazione broker
    initial_cash = 100000.0
    cerebro.broker.setcash(initial_cash)
    cerebro.broker.setcommission(commission=0.0)

    # Analizzatori
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe', riskfreerate=0.02, annualize=True)
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    cerebro.addanalyzer(bt.analyzers.TradeAnalyzer, _name='trade')

    # Run backtest
    print("\nRunning backtest on all assets...\n")
    results = cerebro.run()
    strat = results[0]
    
    final_value = cerebro.broker.getvalue()
    years = (dataframes['SPY'].index[-1] - dataframes['SPY'].index[0]).days / 365.25
    
    # Calcola metriche
    metrics = calculate_metrics(strat, initial_cash, final_value, years)
    
    # Stampa analisi dettagliata
    print_detailed_analysis(metrics, strat.trades)
    
    # Crea grafico completo
    plot_comprehensive_analysis(strat.dates, strat.equity_curve, metrics, strat.trades,
                               'magnificent7_portfolio.jpg')
    
    print("✅ Backtest completato su tutti gli asset!")
    print(f"📊 Portfolio finale: ${final_value:,.2f}")
    print(f"📈 Return totale: {metrics['total_return']:.2f}%")
    print(f"📉 Max Drawdown: {metrics['max_dd']:.2f}%")


1 Failed download:
['SPY']: ValueError('day is out of range for month')

1 Failed download:
['QQQ']: ValueError('day is out of range for month')

1 Failed download:
['AAPL']: ValueError('day is out of range for month')

1 Failed download:
['MSFT']: ValueError('day is out of range for month')

1 Failed download:
['GOOGL']: ValueError('day is out of range for month')

1 Failed download:
['AMZN']: ValueError('day is out of range for month')

1 Failed download:
['NVDA']: ValueError('day is out of range for month')

1 Failed download:
['TSLA']: ValueError('day is out of range for month')

1 Failed download:
['META']: ValueError('day is out of range for month')


                    DOWNLOADING MULTI-ASSET DATA

✓ Loaded 9 assets successfully

Running backtest on all assets...



IndexError: index -1 is out of bounds for axis 0 with size 0

In [7]:
pip install quantstats

                                              0.0/81.4 kB ? eta -:--:--
     ---------------------------------------- 81.4/81.4 kB 4.4 MB/s eta 0:00:00
  Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)
Note: you may need to restart the kernel to use updated packages.
